# 📖 Lab 1: Rate Limiter Placement & Client Identification

Before we can limit requests, we need two decisions:
1. **Where** does the rate limiter live? (in-process, dedicated service, or API gateway)
2. **How** do we identify clients? (user ID, IP, API key)

## 🏗️ Three Placement Options

```
❌ In-Process                    🟡 Dedicated Service              ✅ API Gateway
─────────────                    ────────────────────              ────────────────
Client ──> App Server 1          Client ──> App Server ──> RL      Client ──> Gateway+RL ──> App
           (local counter)                   (network call)                   (edge check)
           App Server 2          Each request = extra hop           Blocked requests
           (local counter)       Full app context available         never reach servers
           ...
           = NO global view!     = latency + failure mode           = limited to HTTP context
```

We choose **API Gateway** — most common in production. Centralized, no extra hops, blocked traffic never reaches app servers.

## Learning Objectives

- Understand the 3 placement options and their trade-offs
- Simulate the in-process problem (no global view)
- Implement client identification from HTTP request context
- Build layered rules (per-user, per-IP, per-endpoint) with most-restrictive enforcement

## 🛠️ Setup

```bash
cd system-designs/rate-limiter
docker-compose up -d
```

Select the **"Rate Limiter (Python)"** kernel.

In [ ]:
import time
import threading
from dataclasses import dataclass

print("✅ Ready!")

## ❌ The In-Process Problem

Let's prove why local-only rate limiting fails with multiple servers. We'll simulate 5 app servers, each with their own local counter, and a user sending requests distributed across them.

In [ ]:
class InProcessRateLimiter:
    """Each server has its own local counter — no global view."""

    def __init__(self, limit: int):
        self.limit = limit
        self.counters: dict[str, int] = {}

    def is_allowed(self, client_id: str) -> bool:
        count = self.counters.get(client_id, 0)
        if count >= self.limit:
            return False
        self.counters[client_id] = count + 1
        return True


# Simulate 5 app servers, each with their own rate limiter
LIMIT = 100  # "100 requests per minute per user"
NUM_SERVERS = 5
TOTAL_REQUESTS = 300  # Alice sends 300 requests total

servers = [InProcessRateLimiter(limit=LIMIT) for _ in range(NUM_SERVERS)]

# Simulate round-robin load balancing
allowed = 0
blocked = 0

for i in range(TOTAL_REQUESTS):
    server = servers[i % NUM_SERVERS]  # round-robin
    if server.is_allowed("alice"):
        allowed += 1
    else:
        blocked += 1

print(f"🎯 Goal: limit Alice to {LIMIT} requests/minute\n")
print(f"  Servers: {NUM_SERVERS}")
print(f"  Requests sent: {TOTAL_REQUESTS}")
print(f"  Allowed: {allowed}")
print(f"  Blocked: {blocked}")

print(f"\n  Per-server view:")
for i, s in enumerate(servers):
    count = s.counters.get("alice", 0)
    print(f"    Server {i+1}: saw {count} requests → thinks Alice is fine (< {LIMIT})")

print(f"\n  🚨 Each server saw only {TOTAL_REQUESTS // NUM_SERVERS} requests.")
print(f"     None of them blocked Alice!")
print(f"     Global total: {allowed} requests allowed — {allowed - LIMIT} OVER the limit!")
print(f"\n  ⚠️  In-process rate limiting has NO global view.")
print(f"     With {NUM_SERVERS} servers, the effective limit is {LIMIT} × {NUM_SERVERS} = {LIMIT * NUM_SERVERS}!")

## 🔍 Client Identification

At the API Gateway, we only have the HTTP request. Let's build a client identifier that extracts identity from request context and supports multiple strategies.

In [ ]:
import base64
import json

@dataclass
class HttpRequest:
    """Simulates an incoming HTTP request at the API Gateway."""
    method: str
    path: str
    headers: dict[str, str]
    remote_ip: str


def extract_client_id(request: HttpRequest) -> dict:
    """
    Extract client identity from HTTP request context.
    Returns all identifiers found — the rate limiter checks rules against each.
    """
    identifiers = {}

    # 1. User ID from JWT token in Authorization header
    auth = request.headers.get("Authorization", "")
    if auth.startswith("Bearer "):
        token = auth.split(" ")[1]
        # In production: verify JWT signature. Here we just decode the payload.
        try:
            payload_b64 = token.split(".")[1]
            payload_b64 += "=" * (4 - len(payload_b64) % 4)  # pad
            payload = json.loads(base64.urlsafe_b64decode(payload_b64))
            identifiers["user_id"] = payload.get("sub")
            identifiers["tier"] = payload.get("tier", "free")
        except Exception:
            pass

    # 2. API Key from X-API-Key header
    api_key = request.headers.get("X-API-Key")
    if api_key:
        identifiers["api_key"] = api_key

    # 3. IP Address (always available)
    ip = request.headers.get("X-Forwarded-For", request.remote_ip)
    identifiers["ip"] = ip.split(",")[0].strip()  # first IP in chain

    # 4. Endpoint (for per-endpoint rules)
    identifiers["endpoint"] = f"{request.method}:{request.path}"

    return identifiers


# Test with different request types
print("🔍 Client identification from HTTP requests:\n")

# Authenticated user with JWT
jwt_payload = base64.urlsafe_b64encode(json.dumps({"sub": "alice_123", "tier": "premium"}).encode()).decode()
fake_jwt = f"header.{jwt_payload}.signature"

requests_examples = [
    HttpRequest("GET", "/api/timeline", {"Authorization": f"Bearer {fake_jwt}"}, "1.2.3.4"),
    HttpRequest("GET", "/api/search", {"X-API-Key": "dev_key_abc123"}, "5.6.7.8"),
    HttpRequest("POST", "/api/tweet", {"X-Forwarded-For": "10.0.0.1, 192.168.1.1"}, "proxy.internal"),
    HttpRequest("GET", "/api/public", {}, "203.0.113.42"),
]

labels = ["Authenticated user (JWT)", "Developer API key", "Behind proxy (X-Forwarded-For)", "Anonymous (IP only)"]

for label, req in zip(labels, requests_examples):
    ids = extract_client_id(req)
    print(f"  📌 {label}")
    for k, v in ids.items():
        print(f"     {k}: {v}")
    print()

## 📋 Layered Rules: Most-Restrictive Wins

Real rate limiters don't just have one rule. They layer multiple rules and enforce **the most restrictive one**. Let's build this.

In [ ]:
@dataclass
class RateLimitRule:
    """A rate limiting rule: N requests per time_window seconds."""
    rule_id: str
    match_field: str    # which client identifier to match on (user_id, ip, endpoint)
    match_value: str    # specific value or "*" for all
    limit: int          # max requests
    window_seconds: int # time window

    def make_key(self, client_ids: dict) -> str:
        """Build the counter key for this rule + client."""
        if self.match_field == "endpoint":
            return f"rl:{self.rule_id}:{client_ids.get('user_id', client_ids.get('ip', 'anon'))}:{client_ids.get('endpoint', '*')}"
        value = client_ids.get(self.match_field, "unknown")
        return f"rl:{self.rule_id}:{value}"

    def applies_to(self, client_ids: dict) -> bool:
        """Does this rule apply to this request?"""
        if self.match_value == "*":
            return self.match_field in client_ids
        return client_ids.get(self.match_field) == self.match_value


# Define our layered rules
rules = [
    RateLimitRule("user_free",    "tier",     "free",    100,  60),   # Free users: 100/min
    RateLimitRule("user_premium", "tier",     "premium", 1000, 60),   # Premium: 1000/min
    RateLimitRule("ip_global",    "ip",       "*",       200,  60),   # Per-IP: 200/min
    RateLimitRule("search_limit", "endpoint", "*",       10,   60),   # Search: 10/min/user
]

# Simple in-memory rate checker
counters: dict[str, list[float]] = {}

def check_rules(client_ids: dict, current_rules: list[RateLimitRule]) -> dict:
    """Check all applicable rules, return the most restrictive result."""
    results = []

    for rule in current_rules:
        if not rule.applies_to(client_ids):
            continue

        # For endpoint-specific rules, only apply if endpoint matches
        if rule.rule_id == "search_limit" and "search" not in client_ids.get("endpoint", ""):
            continue

        key = rule.make_key(client_ids)
        now = time.time()

        # Simple fixed window counter
        if key not in counters:
            counters[key] = []
        # Remove old entries outside the window
        counters[key] = [t for t in counters[key] if t > now - rule.window_seconds]

        remaining = rule.limit - len(counters[key])
        allowed = remaining > 0

        if allowed:
            counters[key].append(now)
            remaining -= 1

        results.append({
            "rule": rule.rule_id,
            "allowed": allowed,
            "remaining": remaining,
            "limit": rule.limit,
        })

    # Most restrictive wins: if ANY rule blocks, the request is blocked
    blocked_by = [r for r in results if not r["allowed"]]
    if blocked_by:
        return {"allowed": False, "blocked_by": blocked_by[0]["rule"], "results": results}
    return {"allowed": True, "results": results}


# Simulate: Alice (premium) makes requests
counters.clear()

print("📋 Layered Rate Limiting Demo:\n")
print("  Rules:")
for r in rules:
    print(f"    {r.rule_id}: {r.limit} req/{r.window_seconds}s (match: {r.match_field}={r.match_value})")

# Build client IDs for Alice (premium user)
alice_request = HttpRequest("GET", "/api/search", {"Authorization": f"Bearer {fake_jwt}"}, "1.2.3.4")
alice_ids = extract_client_id(alice_request)
print(f"\n  Alice's identifiers: {alice_ids}\n")

# Send 15 search requests — should hit the 10/min search limit
print(f"  Sending 15 search requests:")
for i in range(15):
    result = check_rules(alice_ids, rules)
    status = "✅" if result["allowed"] else f"❌ blocked by: {result['blocked_by']}"
    if i < 3 or i >= 9:
        print(f"    Request {i+1:2d}: {status}")
    elif i == 3:
        print(f"    ... (requests 4-9 allowed)")

# Show final state
print(f"\n  📊 Final rule state for Alice:")
for r in check_rules(alice_ids, rules)["results"]:
    status = "✅" if r["allowed"] else "❌ BLOCKED"
    print(f"    {r['rule']}: {r['remaining']}/{r['limit']} remaining — {status}")

print(f"\n  💡 Alice has premium tier (1000/min) but hit the search endpoint limit (10/min).")
print(f"     Most restrictive rule wins!")

## ✅ Summary

### Placement Decision

| Placement | Global View | Latency | Context | Choose When |
|-----------|------------|---------|---------|-------------|
| In-process | ❌ None | ✅ Zero | ✅ Full app context | Single server only |
| Dedicated service | ✅ Yes | ❌ Extra hop per request | ✅ Full app context | Need complex business rules |
| **API Gateway** ✅ | ✅ Yes | ✅ No extra hop | 🟡 HTTP context only | Most production systems |

### Client Identification

| Strategy | Source | Strength | Weakness |
|----------|--------|----------|----------|
| User ID | JWT in `Authorization` | Precise per-user limits | Requires authentication |
| IP Address | `X-Forwarded-For` | Always available | NAT — many users share one IP |
| API Key | `X-API-Key` header | Per-developer limits | Key sharing / theft |

### Layered Rules

Real systems apply **multiple rules** and enforce the most restrictive:
- Per-user tier limits (free vs premium)
- Per-IP limits (protect against anonymous abuse)
- Per-endpoint limits (expensive endpoints get stricter limits)
- Global limits (system-wide protection)

**Next up:** Lab 2 — Rate limiting algorithms (fixed window, sliding window, token bucket, sliding window counters)